# Практична робота №9
## Дисципліна: «Аналіз даних та математичне моделювання»
### Тема: «Перетворення даних: похідні змінні, дискретизація, масштабування, індикаторне кодування»

---

**Студент:** Хрустовський Павло  
**Група:** ІТ-42  
**Варіант:** 2  
- **Місто магазину:** Львів  
- **Базова ціна товару:** 720 грн  
- **Базова кількість у замовленні:** 4  

**Мета заняття:** на наборі клієнтів свого інтернет-магазину (Варіант 2: Львів, базова ціна 720 грн, базова кількість 4) побудувати похідну змінну, розбити числову змінну на групи двома способами (`pd.cut`, `pd.qcut`), привести змінні різного масштабу (`вік` і `дохід`) до спільної шкали (min-max нормалізація та z-стандартизація) й закодувати номінальну категорію (`місто_доставки`) індикаторним способом (one-hot encoding) — практично засвоїти всі чотири ключові методи конструювання та підготовки ознак (feature engineering).

## Завдання 1. Побудова набору клієнтів інтернет-магазину

Згідно з умовами завдання, формуємо `DataFrame` із 10 замовленнями клієнтів інтернет-магазину для **Варіанта 2**:
- `вік` — вік клієнта (значення від 18 до 70 років);
- `дохід` — місячний дохід клієнта у гривнях (свідомо іншого порядку величини $10^4 - 10^5$, ніж вік, для подальшого масштабування);
- `кількість` — кількість одиниць товару в замовленні (орієнтир — базова кількість варіанта 4);
- `місто_доставки` — місто доставки замовлення (використано щонайменше 4 різні міста з таблиці варіантів курсу, зокрема Львів, Київ, Одеса, Івано-Франківськ, Харків, Дніпро).

In [1]:
import pandas as pd
import numpy as np

# Фіксація базових параметрів для Варіанта 2
BASE_CITY = "Львів"
BASE_PRICE = 720  # грн
BASE_QTY = 4

# Створення набору з 10 клієнтів
clients_data = [
    {"вік": 21, "дохід": 22000, "кількість": 3, "місто_доставки": "Львів"},
    {"вік": 24, "дохід": 35000, "кількість": 4, "місто_доставки": "Київ"},
    {"вік": 26, "дохід": 28000, "кількість": 4, "місто_доставки": "Львів"},
    {"вік": 29, "дохід": 45000, "кількість": 5, "місто_доставки": "Одеса"},
    {"вік": 33, "дохід": 32000, "кількість": 2, "місто_доставки": "Івано-Франківськ"},
    {"вік": 36, "дохід": 54000, "кількість": 4, "місто_доставки": "Львів"},
    {"вік": 42, "дохід": 38000, "кількість": 6, "місто_доставки": "Харків"},
    {"вік": 49, "дохід": 62000, "кількість": 4, "місто_доставки": "Львів"},
    {"вік": 58, "дохід": 25000, "кількість": 3, "місто_доставки": "Київ"},
    {"вік": 67, "дохід": 48000, "кількість": 5, "місто_доставки": "Дніпро"},
]

clients = pd.DataFrame(clients_data)
print(f"Створено таблицю клієнтів ({len(clients)} рядків). Базове місто: {BASE_CITY}, ціна: {BASE_PRICE} грн.")
clients

Створено таблицю клієнтів (10 рядків). Базове місто: Львів, ціна: 720 грн.


,вік,дохід,кількість,місто_доставки
0,21,22000,3,Львів
1,24,35000,4,Київ
2,26,28000,4,Львів
3,29,45000,5,Одеса
4,33,32000,2,Івано-Франківськ
5,36,54000,4,Львів
6,42,38000,6,Харків
7,49,62000,4,Львів
8,58,25000,3,Київ
9,67,48000,5,Дніпро


## Завдання 2. Похідні змінні

Обчислюємо два похідні стовпці:
1. `сума_замовлення` = базова ціна товару (720 грн) $\times$ `кількість`.
2. `частка_доходу` = `сума_замовлення` / `дохід`.

In [2]:
# Обчислення похідних змінних
clients["сума_замовлення"] = BASE_PRICE * clients["кількість"]
clients["частка_доходу"] = clients["сума_замовлення"] / clients["дохід"]

# Виведення результатів з форматуванням
display_task2 = clients[["вік", "дохід", "кількість", "сума_замовлення", "частка_доходу"]].copy()
display_task2["частка_доходу_%"] = (display_task2["частка_доходу"] * 100).round(2).astype(str) + "%"
display_task2

,вік,дохід,кількість,сума_замовлення,частка_доходу,частка_доходу_%
0,21,22000,3,2160,0.098182,9.82%
1,24,35000,4,2880,0.082286,8.23%
2,26,28000,4,2880,0.102857,10.29%
3,29,45000,5,3600,0.080000,8.0%
4,33,32000,2,1440,0.045000,4.5%
5,36,54000,4,2880,0.053333,5.33%
6,42,38000,6,4320,0.113684,11.37%
7,49,62000,4,2880,0.046452,4.65%
8,58,25000,3,2160,0.086400,8.64%
9,67,48000,5,3600,0.075000,7.5%


### Письмове пояснення до Завдання 2

**Чому `частка_доходу` як окрема змінна може бути корисніша для аналізу поведінки клієнтів, ніж `сума_замовлення` і `дохід`, розглянуті окремо один від одного?**

1. **Нормування купівельної спроможності (відносне фінансове навантаження):**
   - Сама по собі `сума_замовлення` відображає лише абсолютний чек у касі магазину. Наприклад, замовлення на 2880 грн є абсолютно однаковим чеком як для клієнта №2 (дохід 28 000 грн), так і для клієнта №7 (дохід 62 000 грн).
   - Проте для клієнта з доходом 28 000 грн це замовлення становить **10.29%** місячного бюджету, що є вагомою покупкою, яка вимагає обмірковування. Для клієнта з доходом 62 000 грн ця ж сума становить лише **4.65%** доходу — тобто є практично непомітною буденною витратою.
   - Окремий стовпець `дохід` показує лише загальну заможність особи, але нічого не говорить про її поточні витрати в цьому магазині.

2. **Інсайт про психологію споживача та цінову чутливість:**
   - Клієнти з високою `частка_доходу` (понад 10–15%) є високочутливими до цінових коливань, знижок, програм лояльності та вартості доставки. Для їхнього утримання ефективні розстрочки або бонуси.
   - Клієнти з низькою `частка_доходу` менш чутливі до ціни: для них мають першочергове значення швидкість доставки, сервіс і якість; вони є ідеальними кандидатами для додаткових продажів (cross-selling та up-selling).

3. **Спрощення навчання для алгоритмів (Feature Engineering):**
   - Більшість лінійних моделей (наприклад, логістична регресія, лінійні SVM) шукають лінійні комбінації ознак $w_1 x_1 + w_2 x_2$. Вони неспроможні самостійно вивести операцію ділення $\frac{x_1}{x_2}$.
   - Створюючи `частка_доходу` явно, ми передаємо моделі готовий нелінійний поведінковий фактор, що суттєво підвищує роздільну здатність алгоритмів класифікації та кластеризації.

## Завдання 3. Дискретизація числової змінної (вік)

Здійснюємо дискретизацію віку клієнтів на 3 групи двома підходами:
1. `pd.cut()` — інтервали однакової довжини (Equal Width Binning);
2. `pd.qcut()` — квантилі з однаковою кількістю клієнтів (Equal Frequency Binning).

In [3]:
# 1. pd.cut — інтервали однакової ширини за шкалою значень віку
clients["вік_група_cut"] = pd.cut(clients["вік"], bins=3)
cut_counts = clients["вік_група_cut"].value_counts().sort_index()

print("--- 1. pd.cut() (інтервали однакової довжини) ---")
print(cut_counts)

# 2. pd.qcut — квантилі з однаковою кількістю спостережень
clients["вік_група_qcut"] = pd.qcut(clients["вік"], q=3)
qcut_counts = clients["вік_група_qcut"].value_counts().sort_index()

print("\n--- 2. pd.qcut() (однакова кількість спостережень у групах) ---")
print(qcut_counts)

--- 1. pd.cut() (інтервали однакової довжини) ---
вік_група_cut
(20.954, 36.333]    6
(36.333, 51.667]    2
(51.667, 67.0]      2
Name: count, dtype: int64

--- 2. pd.qcut() (однакова кількість спостережень у групах) ---
вік_група_qcut
(20.999, 29.0]    4
(29.0, 42.0]      3
(42.0, 67.0]      3
Name: count, dtype: int64


### Порівняння результатів та письмове пояснення до Завдання 3

**Чому кількість клієнтів у групах відрізняється між `cut` і `qcut`, хоча вхідні дані ті самі?**

1. **Принцип роботи `pd.cut()` (Equal Width):**
   - Функція бере повний числовий розмах змінної $\Delta = X_{max} - X_{min} = 67 - 21 = 46$ років і розбиває його на 3 інтервали строго однакової довжини: $\frac{46}{3} \approx 15.33$ роки.
   - Межі інтервалів фіксовані за значеннями:
     - `(20.954, 36.333]` $\rightarrow$ потрапило **6 клієнтів** (21, 24, 26, 29, 33, 36);
     - `(36.333, 51.667]` $\rightarrow$ потрапило **2 клієнти** (42, 49);
     - `(51.667, 67.0]` $\rightarrow$ потрапило **2 клієнти** (58, 67).
   - Оскільки реальний розподіл віку є скошеним (більшість клієнтів магазину є молодими людьми до 36 років), інтервали однакової довжини виявилися вкрай нерівномірно наповненими (6, 2, 2).

2. **Принцип роботи `pd.qcut()` (Equal Frequency):**
   - Функція сортує спостереження за зростанням і розбиває вибірку на квантилі так, щоб у кожному кошику була приблизно однакова частка клієнтів (по $\approx 33.3\%$ або 4, 3, 3 осіб для вибірки з 10 елементів).
   - Межі підлаштовуються під емпіричну густину розподілу:
     - `(20.999, 29.0]` $\rightarrow$ **4 клієнти** (ширина інтервалу лише $\approx 8$ років, оскільки тут висока щільність точок);
     - `(29.0, 42.0]` $\rightarrow$ **3 клієнти** (ширина інтервалу 13 років);
     - `(42.0, 67.0]` $\rightarrow$ **3 клієнти** (ширина інтервалу 25 років, оскільки старші клієнти зустрічаються рідше).

**Висновок:**  
`cut` гарантує сталу ширину інтервалів на числовій осі, але призводить до різної кількості спостережень при нерівномірному розподілі. `qcut` гарантує рівномірне наповнення кошиків спостереженнями, але формує інтервали різної ширини, звужуючи їх у зонах скупчення даних і розширюючи в зонах розрідження.

## Завдання 4. Масштабування змінних (вік та дохід)

Застосовуємо два стандартизовані методи масштабування до змінних різного порядку величини (`вік` $\sim 10^1$, `дохід` $\sim 10^4 - 10^5$):
1. **Min-Max нормалізація:**
   $$X_{norm} = \frac{X - X_{min}}{X_{max} - X_{min}} \in [0, 1]$$
2. **Z-оцінка (стандартизація):**
   $$X_{z} = \frac{X - \bar{X}}{s}, \quad \text{де } \bar{X} \text{ — вибіркове середнє, } s \text{ — вибіркове стандартне відхилення}$$

In [4]:
# Вихідні серії
age = clients["вік"]
income = clients["дохід"]

# 1. Min-Max нормалізація у діапазон [0, 1]
clients["вік_norm"] = (age - age.min()) / (age.max() - age.min())
clients["дохід_norm"] = (income - income.min()) / (income.max() - income.min())

# 2. Z-оцінка (стандартизація до mean=0, std=1)
clients["вік_z"] = (age - age.mean()) / age.std()
clients["дохід_z"] = (income - income.mean()) / income.std()

# Порівняльна таблиця вихідних та масштабованих значень
comparison_df = clients[[
    "вік", "вік_norm", "вік_z",
    "дохід", "дохід_norm", "дохід_z"
]]
display(comparison_df.head(6))

# Перевірка статистичних властивостей масштабованих змінних
print("\nСтатистичні показники після масштабування:")
print(comparison_df.describe().round(3).loc[["min", "mean", "std", "max"]])

,вік,вік_norm,вік_z,дохід,дохід_norm,дохід_z
0,21,0.000000,-1.141709,22000,0.000,-1.290741
1,24,0.065217,-0.945988,35000,0.325,-0.297863
2,26,0.108696,-0.815506,28000,0.150,-0.832490
3,29,0.173913,-0.619785,45000,0.575,0.465889
4,33,0.260870,-0.358823,32000,0.250,-0.526989
5,36,0.326087,-0.163101,54000,0.800,1.153266



Статистичні показники після масштабування:
         вік  вік_norm  вік_z      дохід  дохід_norm  дохід_z
min   21.000     0.000 -1.142  22000.000       0.000   -1.291
mean  38.500     0.380 -0.000  38900.000       0.423   -0.000
std   15.328     0.333  1.000  13093.255       0.327    1.000
max   67.000     1.000  1.859  62000.000       1.000    1.764


### Письмове пояснення до Завдання 4

**Чому до масштабування дохід і вік не можна було б напряму порівнювати чи об'єднувати в одному методі, чутливому до масштабу?**

1. **Несумірність шкал та порядків величин:**
   - Вік вимірюється в роках і варіюється від 21 до 67 (розмах 46 років, порядок величини $10^1$).
   - Дохід вимірюється в гривнях і становить від 22 000 до 62 000 грн (розмах 40 000 грн, порядок величини $10^4 - 10^5$).
   - Різниця між максимальним і мінімальним доходом у $\approx 870$ разів більша за різницю у віці.

2. **Спотворення дистанційних метрик (k-means, k-NN, SVM, ієрархічна кластеризація):**
   - У метричних алгоритмах відстань між двома клієнтами $A$ та $B$ розраховується за евклідовою формулою:
     $$d(A, B) = \sqrt{(\text{вік}_A - \text{вік}_B)^2 + (\text{дохід}_A - \text{дохід}_B)^2}$$
   - Наприклад, якщо клієнти різняться за віком на 30 років, внесок віку становить $30^2 = 900$. Якщо вони водночас різняться за доходом на 10 000 грн, внесок доходу становить $10\,000^2 = 100\,000\,000$.
   - Частка доходу у квадраті відстані становитиме $\frac{100\,000\,000}{100\,000\,900} \approx 99.9991\%$. Вік практично повністю нівелюється, і алгоритм кластеризуватиме клієнтів виключно за доходами.

3. **Вплив на регуляризовані регресійні моделі (Ridge, Lasso):**
   - Функція втрат із регуляризацією штрафує величину вагових коефіцієнтів $\lambda \sum w_j^2$ або $\lambda \sum |w_j|$.
   - Оскільки ознака доходу має гігантські значення, її коефіцієнт $w_{\text{дохід}}$ природно буде мізерно малим (наприклад, $10^{-5}$), тому регуляризація його майже не штрафуватиме. Ознака віку, навпаки, матиме більший коефіцієнт і зазнає необґрунтованого заниження.

4. **Результат масштабування:**
   - Min-max стискає обидва параметри строго у відрізок $[0, 1]$.
   - Z-оцінка центрує середнє у $0$ і масштабує розкид до одиничної дисперсії ($\sigma = 1$).
   - Обидві змінні стають безрозмірнісними та вносять рівноправний вклад у роботу чутливих моделей.

## Завдання 5. Індикаторне кодування (One-Hot Encoding)

Закодуємо номінальну змінну `місто_доставки` за допомогою `pd.get_dummies()`, долучимо отримані стовпці до основної таблиці та перевіримо відповідність кількості створених стовпців кількості унікальних міст.

In [5]:
# 1. One-hot кодування номінальної змінної 'місто_доставки'
city_dummies = pd.get_dummies(clients["місто_доставки"], prefix="місто", dtype=int)

# 2. Долучення індикаторних стовпців до основної таблиці
clients = pd.concat([clients, city_dummies], axis=1)

# 3. Перевірка відповідності кількості колонок
unique_cities_count = clients["місто_доставки"].nunique()
dummy_columns_count = len(city_dummies.columns)

print(f"Кількість унікальних міст у даних: {unique_cities_count}")
print(f"Кількість створених індикаторних стовпців: {dummy_columns_count}")
print("Список унікальних міст:", list(clients["місто_доставки"].unique()))

# Твердження assert для перевірки умови завдання
assert unique_cities_count == dummy_columns_count, "Помилка: кількість стовпців не збігається!"
print("Перевірка успішна: кількість нових стовпців дорівнює кількості унікальних міст!")

# Виведення результату
display_cols = ["місто_доставки"] + list(city_dummies.columns)
clients[display_cols]

Кількість унікальних міст у даних: 6
Кількість створених індикаторних стовпців: 6
Список унікальних міст: ['Львів', 'Київ', 'Одеса', 'Івано-Франківськ', 'Харків', 'Дніпро']
Перевірка успішна: кількість нових стовпців дорівнює кількості унікальних міст!


,місто_доставки,місто_Івано-Франківськ,місто_Дніпро,місто_Київ,місто_Львів,місто_Одеса,місто_Харків
0,Львів,0,0,0,1,0,0
1,Київ,0,0,1,0,0,0
2,Львів,0,0,0,1,0,0
3,Одеса,0,0,0,0,1,0
4,Івано-Франківськ,1,0,0,0,0,0
5,Львів,0,0,0,1,0,0
6,Харків,0,0,0,0,0,1
7,Львів,0,0,0,1,0,0
8,Київ,0,0,1,0,0,0
9,Дніпро,0,1,0,0,0,0


### Письмове пояснення до Завдання 5

**Чому для `місто_доставки` доречний саме one-hot, а не порядкове кодування (мапінг міст у 1, 2, 3, ...)?**

1. **Номінальний (якісний) тип змінної:**
   - Змінна `місто_доставки` є номінальною: міста не мають природного ієрархічного чи кількісного порядку. Львів не є «більшим» або «кращим» за Київ, а Одеса не передує Харкову в сенсі математичних відношень переваги.

2. **Небезпека та артефакти порядкового кодування (Ordinal Encoding):**
   - Якщо призначити кожному місту ціле число (наприклад, Львів = 1, Київ = 2, Одеса = 3, Харків = 4, Івано-Франківськ = 5, Дніпро = 6), будь-яка математична модель сприйматиме ці значення як точки на числовій осі.
   - Це вносить серйозні хибні припущення:
     - **Хибний порядок:** Модель вважатиме, що Дніпро (6) є «більшим» за Львів (1) у 6 разів.
     - **Хибні відстані:** Відстань між Львовом (1) і Києвом (2) дорівнюватиме $|2 - 1| = 1$, а між Львовом (1) і Дніпром (6) — $|6 - 1| = 5$. Алгоритм зробить висновок, що Київ у 5 разів «ближчий» до Львова, ніж Дніпро, що є суто артефактом обраної нумерації.
     - **Хибна арифметика:** Середнє між Львовом (1) і Одесою (3) дасть $(1 + 3) / 2 = 2$ (Київ), що не має жодного фізичного змісту.

3. **Перевага One-Hot кодування:**
   - One-hot перетворює кожне місто на окремий ортогональний бінарний вектор у просторі розмірності $k$ (де $k$ — кількість міст):
     - Львів: `[1, 0, 0, 0, 0, 0]`
     - Київ: `[0, 1, 0, 0, 0, 0]`
     - Одеса: `[0, 0, 1, 0, 0, 0]`
   - Евклідова відстань між будь-якою парою різних міст $A$ і $B$ є строго однаковою:
     $$d(A, B) = \sqrt{(1 - 0)^2 + (0 - 1)^2 + 0 + \dots} = \sqrt{2} \approx 1.414$$
   - Таким чином, модель трактує всі міста як рівноправні незалежні категорії без надуманого взаємного упорядкування.

## Підсумки та вичерпні відповіді на контрольні запитання

### 1. Чи можете пояснити різницю між `pd.cut()` і `pd.qcut()` — коли інтервали однакової довжини, а коли гарантована однакова кількість спостережень?
- **`pd.cut()` (Equal-Width Binning):** фіксує **довжину інтервалів**. Загальний розмах значень $[X_{min}, X_{max}]$ ділиться на $k$ рівних за шириною числових відрізків $\Delta = (X_{max} - X_{min}) / k$. Кількість спостережень у кожному інтервалі залежить від форми розподілу: при наявності асиметрії чи кластерів одні групи будуть переповненими, а інші — порожніми. Застосовується, коли бізнес-логіка вимагає фіксованих діапазонів (наприклад, вікові групи: 18–30, 31–45, 46–60 років).
- **`pd.qcut()` (Equal-Frequency Binning):** фіксує **кількість спостережень** (квантилі/перцентилі). Вибірка ділиться на $k$ груп так, щоб у кожній групі опинилася однакова кількість об'єктів ($N / k$). Ширина числових інтервалів при цьому варіюється: там, де щільність даних висока, інтервали вузькі; у розріджених зонах — широкі. Застосовується для усунення перекосів розподілу та формування збалансованих вибірок для аналізу.

### 2. Чи розумієте, чому для номінальної змінної (місто) застосовують one-hot кодування, а не порядкове чи просто числове (1, 2, 3)?
- Номінальні змінні позбавлені природного відношення порядку. Якщо закодувати їх числами $1, 2, 3, \dots$, алгоритми сприйматимуть ці числа як кількісні величини на інтервальній шкалі, штучно створюючи неіснуючу ієрархію («місто 3 більше за місто 1») та нерівні відстані між категоріями.
- One-hot кодування розгортає категорії в ортогональні бінарні вектори, гарантуючи, що всі категорії рівновіддалені одна від одної (відстань $\sqrt{2}$) і жодна категорія не домінує за шкалою.

### 3. Чи можете пояснити, навіщо взагалі масштабувати змінні перед методом, чутливим до масштабу (дистанційні методи, регресія з регуляризацією)?
- Якщо ознаки мають різні одиниці вимірювання та порядки величин (наприклад, вік $\sim 10^1$ і дохід $\sim 10^5$), ознака з більшим масштабом повністю домінуватиме в обчисленні евклідових чи манхеттенських відстаней ($k$-means, $k$-NN, SVM), зводячи вплив ознак з меншим масштабом практично до нуля.
- У лінійних моделях з регуляризацією (Ridge $L_2$, Lasso $L_1$) штраф застосовується безпосередньо до величини вагових коефіцієнтів. Ознаки з великими числовими значеннями мають штучно занижені коефіцієнти і несправедливо уникають штрафування. Масштабування (min-max нормалізація або z-стандартизація) усуває розмірність і приводить ознаки до єдиного масштабу.

### 4. Чи можете пояснити, чому похідна змінна (наприклад, частка доходу на замовлення) часто несе більше сенсу для аналізу, ніж вихідні складники окремо?
- Вихідні змінні фіксують абсолютні показники ізольовано (`сума` — вартість кошика, `дохід` — фінансовий стан клієнта). Похідна змінна (`частка_доходу`) є коефіцієнтом відносного навантаження і відображає поведінковий зміст: чи є покупка значною подією для клієнта, чи незначною буденною витратою.
- Багато алгоритмів машинного навчання неспроможні самостійно вивчити операцію нелінійного відношення ($x_1 / x_2$). Створення таких ознак людиною-аналітиком (feature engineering) збагачує простір ознак доменним знанням і суттєво підвищує прогнозну силу моделей.